In [ ]:
import pandas as pd
import json
from collections import Counter
from tqdm import tqdm
import numpy as np

# Load audio features
audio_df = pd.read_csv('audio_features_clean.csv')
print(f"Loaded {len(audio_df):,} tracks with emotions")

# Show current emotion distribution
print(f"\nCurrent emotion distribution in audio features:")
print(audio_df['emotion'].value_counts())

# Create fast lookup dictionaries for exact matching
print("\nCreating fast track lookup...")
exact_match_lookup = {}
track_name_lookup = {}

for _, row in audio_df.iterrows():
    track_name = str(row['track_name']).lower().strip()
    artist_name = str(row['artist_name']).lower().strip()
    
    # Extract primary artist (first one if multiple)
    if ',' in artist_name:
        primary_artist = artist_name.split(',')[0].strip()
    else:
        primary_artist = artist_name
    
    # Create exact match key
    exact_key = f"{track_name}|{primary_artist}"
    exact_match_lookup[exact_key] = row.to_dict()
    
    # Store by track name for fuzzy matching
    if track_name not in track_name_lookup:
        track_name_lookup[track_name] = []
    track_name_lookup[track_name].append(row.to_dict())

print(f"Created {len(exact_match_lookup):,} exact match entries")

# Load listening data
print("\nLoading listening data...")
with open('music_listening_data.json', 'r') as f:
    listening_data = json.load(f)
print(f"Loaded {len(listening_data):,} events")

# Function to safely get string value
def safe_str(value):
    """Safely convert to string, handling None, float, etc."""
    if value is None or pd.isna(value):
        return ""
    if isinstance(value, float):
        return ""
    return str(value).lower().strip()

# Process with improved matching
print("\nProcessing tracks...")
processed_events = []
match_stats = {'exact': 0, 'track_only': 0, 'artist_filtered': 0, 'no_match': 0}
error_count = 0

for event in tqdm(listening_data, desc="Processing"):
    try:
        # Safely get track and artist names
        track_name = safe_str(event.get('title', event.get('track_name', '')))
        artist_name = safe_str(event.get('text', event.get('artist', event.get('artist_name', ''))))
        
        # Skip if track name is empty
        if not track_name:
            match_stats['no_match'] += 1
            event['emotion'] = 'unknown'
            event['match_confidence'] = 0
            event['match_type'] = 'empty_track'
            processed_events.append(event)
            continue
        
        # Extract primary artist (take first part before comma)
        if artist_name and ',' in artist_name:
            primary_artist = artist_name.split(',')[0].strip()
        else:
            primary_artist = artist_name
        
        # Try exact match first
        exact_key = f"{track_name}|{primary_artist}"
        
        if exact_key in exact_match_lookup:
            # Perfect match found
            track_info = exact_match_lookup[exact_key]
            match_stats['exact'] += 1
            event['emotion'] = track_info['emotion']
            event['match_confidence'] = 1.0
            event['match_type'] = 'exact'
            event['matched_track'] = track_info['track_name']
            event['matched_artist'] = track_info['artist_name']
            
        elif track_name in track_name_lookup:
            # Match by track name only, then filter by artist
            candidates = track_name_lookup[track_name]
            
            # Try to find best artist match
            best_match = None
            best_score = 0
            
            for candidate in candidates:
                candidate_artist = str(candidate['artist_name']).lower().strip()
                if ',' in candidate_artist:
                    candidate_artist = candidate_artist.split(',')[0].strip()
                
                # Check if artists match
                if primary_artist and primary_artist == candidate_artist:
                    best_match = candidate
                    best_score = 1.0
                    break
                elif primary_artist and (primary_artist in candidate_artist or candidate_artist in primary_artist):
                    best_score = 0.8
                    best_match = candidate
                elif not primary_artist:
                    # No artist info, use track match only
                    best_score = 0.6
                    best_match = candidate
            
            if best_match:
                match_stats['artist_filtered'] += 1
                event['emotion'] = best_match['emotion']
                event['match_confidence'] = best_score
                event['match_type'] = 'artist_filtered'
                event['matched_track'] = best_match['track_name']
                event['matched_artist'] = best_match['artist_name']
            else:
                # Use first match with lower confidence
                match_stats['track_only'] += 1
                event['emotion'] = candidates[0]['emotion']
                event['match_confidence'] = 0.5
                event['match_type'] = 'track_only'
                event['matched_track'] = candidates[0]['track_name']
                event['matched_artist'] = candidates[0]['artist_name']
        else:
            match_stats['no_match'] += 1
            event['emotion'] = 'unknown'
            event['match_confidence'] = 0
            event['match_type'] = 'no_match'
        
        processed_events.append(event)
        
    except Exception as e:
        error_count += 1
        if error_count <= 5:  # Show first 5 errors
            print(f"\nError processing event: {e}")
            print(f"   Event: {event}")
        event['emotion'] = 'error'
        event['match_type'] = 'error'
        processed_events.append(event)

# Results
print(f"\nMatching Results:")
total = len(processed_events)
print(f"   Exact matches:       {match_stats['exact']:6,} ({match_stats['exact']/total*100:.1f}%)")
print(f"   Artist filtered:     {match_stats['artist_filtered']:6,} ({match_stats['artist_filtered']/total*100:.1f}%)")
print(f"   Track name only:     {match_stats['track_only']:6,} ({match_stats['track_only']/total*100:.1f}%)")
print(f"   No matches:          {match_stats['no_match']:6,} ({match_stats['no_match']/total*100:.1f}%)")
print(f"   Errors:              {error_count:6,} ({error_count/total*100:.1f}%)")
print(f"   Total matched:       {match_stats['exact'] + match_stats['artist_filtered'] + match_stats['track_only']:6,} ({(match_stats['exact'] + match_stats['artist_filtered'] + match_stats['track_only'])/total*100:.1f}%)")

# Emotion distribution (excluding unknown and errors)
emotions = [e['emotion'] for e in processed_events if e['emotion'] not in ['unknown', 'error']]
print(f"\nEmotion Distribution (from {len(emotions):,} matched tracks):")
if emotions:
    emotion_counts = Counter(emotions)
    for emotion, count in emotion_counts.most_common():
        print(f"   {emotion:12}: {count:6,} ({count/len(emotions)*100:.1f}%)")
else:
    print("   No matches found!")

# Check what's in the unknown tracks
unknown_tracks = [e for e in processed_events if e['emotion'] == 'unknown'][:10]
if unknown_tracks:
    print(f"\nSample of unknown tracks (first 10):")
    for i, event in enumerate(unknown_tracks):
        track_name = safe_str(event.get('title', event.get('track_name', '')))
        artist_name = safe_str(event.get('text', event.get('artist', '')))
        print(f"   {i+1}. '{track_name}' by '{artist_name}'")

# Save results
output_file = 'processed_listening_data_with_emotions.json'
with open(output_file, 'w') as f:
    json.dump(processed_events, f, indent=2)
print(f"\nSaved {len(processed_events):,} events to {output_file}")

# Show sample of matched tracks
print("\nSample matched tracks:")
matched_events = [e for e in processed_events if e['emotion'] not in ['unknown', 'error']][:10]
for i, event in enumerate(matched_events):
    track_name = safe_str(event.get('title', event.get('track_name', '')))
    artist_name = safe_str(event.get('text', event.get('artist', '')))
    print(f"\n{i+1}. '{track_name}' by '{artist_name}'")
    print(f"   → Emotion: {event['emotion']} (confidence: {event.get('match_confidence', 0):.1f}, type: {event.get('match_type', 'unknown')})")
    if 'matched_track' in event:
        print(f"   → Matched to: '{event['matched_track']}' by '{event['matched_artist']}'")

Loaded 95,235 tracks with emotions

Current emotion distribution in audio features:
emotion
chill        13112
calm         12509
romantic     12306
happy        12196
energetic    11909
sad          11740
focus        11313
party        10150
Name: count, dtype: int64

Creating fast track lookup...
Created 90,696 exact match entries

Loading listening data...
Loaded 93,653 events

Processing tracks...


Processing: 100%|██████████| 93653/93653 [00:00<00:00, 106453.21it/s]



Matching Results:
   Exact matches:       93,569 (99.9%)
   Artist filtered:          2 (0.0%)
   Track name only:          0 (0.0%)
   No matches:              82 (0.1%)
   Errors:                   0 (0.0%)
   Total matched:       93,571 (99.9%)

Emotion Distribution (from 93,571 matched tracks):
   sad         : 17,781 (19.0%)
   romantic    : 13,827 (14.8%)
   energetic   : 13,387 (14.3%)
   party       : 11,736 (12.5%)
   happy       : 10,145 (10.8%)
   chill       :  9,832 (10.5%)
   focus       :  8,811 (9.4%)
   calm        :  8,052 (8.6%)

Sample of unknown tracks (first 10):
   1. '' by ''
   2. '' by ''
   3. '' by ''
   4. '' by ''
   5. '' by ''
   6. '' by ''
   7. '' by ''
   8. '' by ''
   9. '' by ''
   10. '' by ''

Saved 93,653 events to processed_listening_data_with_emotions.json

Sample matched tracks:

1. 'cat hacks (bonus track)' by 'lemon demon'
   → Emotion: romantic (confidence: 1.0, type: exact)
   → Matched to: 'Cat Hacks (Bonus Track)' by 'Lemon Demon'

2.